In [ ]:
import getpass
import os
from dotenv import load_dotenv

load_dotenv('../.env')
openai_api_key = os.getenv('OPENAI_ACCESS_KEY')
huggingface_access_key = os.getenv("HUGGINGFACEHUB_API_TOKEN")

In [ ]:
from typing import List, Literal, Union
import datetime
from typing import Literal, Optional, Tuple

from langchain_core.pydantic_v1 import BaseModel, Field

class Filter(BaseModel):
    field: Literal["Disease", "Gene", "Protein", "DNAMutation", "ProteinMutation", "SNP", "Cell_type", "Drug", "Sign_symptom", "Biological_structure", "Date", 
                   "Duration", "Time", "Frequency", "Severity", "Lab_value", "Dosage", "Diagnostic_procedure", "Therapeutic_procedure", 
                   "Medication", "Clinical_event", "Outcome", "History", "Subject", "Family_history", "Detailed_description", "Area"]
    
    comparison: Literal["eq", "lt", "lte", "gt", "gte"]
    value: Union[str] = Field(
        ...,
    description="",
    )


class Search(BaseModel):
    """Search over a database of clinical trials eligibility criteria"""

    content_search: str = Field(
        ...,
        description="Similarity search query applied to clinical trials eligibility criteria",
    )
    
    filters: List[Filter] = Field(
        default_factory=list,
        description="Filters over specific metadata fields. Final condition is a logical conjunction of all filters.",
    )

    def pretty_print(self) -> None:
        for field in self.__fields__:
            if getattr(self, field) is not None and getattr(self, field) != getattr(
                self.__fields__[field], "default", None
            ):
                print(f"{field}: {getattr(self, field)}")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

system = """You are an expert at converting user questions into database queries. \
You have access to a database of tutorial videos about a software library for building LLM-powered applications. \
Given a question, return a database query optimized to retrieve the most relevant results.

If there are acronyms or words you are not familiar with, do not try to rephrase them."""
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)
llm = ChatOpenAI(model="gpt-3.5-turbo-0125", temperature=0)
structured_llm = llm.with_structured_output(TutorialSearch)
query_analyzer = prompt | structured_llm

In [ ]:
query_analyzer.invoke({"question": ""}).pretty_print()

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "HuggingFaceH4/zephyr-7b-beta"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config)
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
from typing import List

from langchain.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field, validator
from langchain_openai import ChatOpenAI

In [ ]:
 ChatOpenAI(model="gpt-3.5-turbo-0125", temperature=0)

In [ ]:
from langchain_core.pydantic_v1 import BaseModel, Field

class ExpandedQuery(BaseModel):
    """You have performed query expansion to generate a contextual query from a keyword."""

    expanded_query: str = Field(
        ...,
        description="A concise query generated from the provided keyword embedded in a meaningful context.",
    )
    
from langchain.output_parsers import PydanticToolsParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

system = """You are an expert at converting user-supplied keywords into helpful and concise database queries. Perform query expansion. The keyword is a piece of a cancer patient information. You need to embed it in meaningful context and expand it to a query.
Do not try to rephrase the keyword. Do not add extra information. Just embed the keyword in one short sentence. 

Examples:

1- Keyword: BRCA, Expanded Query: The patient has BRCA mutation.

2- Keyword: EGFR, Expanded Query: The patient's genetic profile shows an EGFR mutation.

3- Keyword: HER2, Expanded Query: The patient's HER2 status is positive.

4- Keyword: ALK, Expanded Query: The patient has ALK rearrangement.

5- Keyword: PD-L1, Expanded Query: The patient has PD-L1 expression.

Keyword: {question}

Expanded Query:
"""
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)
llm = ChatOpenAI(model="gpt-3.5-turbo-0125", temperature=0)
llm_with_tools = llm.bind_tools([ExpandedQuery])
query_analyzer = prompt | llm_with_tools | PydanticToolsParser(tools=[ExpandedQuery])


In [ ]:
query_analyzer.invoke(
                      {"question": "BRCA"},)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Set random seed for reproducibility
torch.random.manual_seed(0)

# Load the pre-trained model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-128k-instruct", 
    device_map="cuda", 
    torch_dtype="auto", 
    trust_remote_code=True, 
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-128k-instruct")

# Define the text generation pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

# Generation arguments
generation_args = {
    "max_new_tokens": 100,  # Adjust as needed for sentence length
    "return_full_text": True,
    "temperature": 0.5,    # Adjust for more creative outputs
    "do_sample": True,     # Enable sampling to allow for diverse outputs
}

In [ ]:
# Install transformers from source - only needed for versions <= v4.34
# pip install git+https://github.com/huggingface/transformers.git
# pip install accelerate

import torch
from transformers import pipeline

pipe = pipeline("text-generation", model="HuggingFaceH4/zephyr-7b-alpha", torch_dtype=torch.bfloat16, device_map="auto")

# We use the tokenizer's chat template to format each message - see https://huggingface.co/docs/transformers/main/en/chat_templating
messages = [
    {
        "role": "system",
        "content": "You are a clinical assistant responsible for creating sentences from keywords. These keywords are related to a patient's medical conditions. You need to generate a sentence that includes the keyword in a meaningful context that helps in querying a database.",
    },
    {"role": "user", "content": "KRAS"},
]
prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
outputs = pipe(prompt, max_new_tokens=256, do_sample=True, temperature=0.7, top_k=50, top_p=0.95)
print(outputs[0]["generated_text"])

In [ ]:
expanded_sentence